# 2. Exploración de Silver

Propósito: Explorar el schema normalizado (modelo estrella) en silver.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6
JAVA_HOME: C:\Program Files\Java\jdk-21


In [2]:
from pyspark.sql import SparkSession, functions as F
from app.utils.spark import SparkClient

# Reutiliza la sesion activa del kernel; si no hay, crea una con la config
# Windows-correcta de SparkClient (rutas nativas Hadoop, memoria del driver).
spark = SparkSession.getActiveSession() or SparkClient().get_session()
silver_path = "data/silver"

In [3]:
import os
files = [f for f in os.listdir(silver_path) if f.endswith('.parquet')]
print("Tablas disponibles en silver:")
for f in sorted(files):
    print(f"  {f}")

Tablas disponibles en silver:
  DIM_ANIO_APLICACION.parquet
  DIM_EJECUTORA.parquet
  DIM_ESPECIFICA.parquet
  DIM_FORMULARIO_SISMEPRE.parquet
  DIM_FUENTE_FINANCIAMIENTO.parquet
  DIM_GENERICA.parquet
  DIM_NIVEL_GOBIERNO.parquet
  DIM_PLIEGO.parquet
  DIM_PREGUNTA_RENAMU.parquet
  DIM_PREGUNTA_SISMEPRE.parquet
  DIM_RUBRO.parquet
  DIM_SECTOR.parquet
  DIM_TIEMPO.parquet
  DIM_TIPO_RECURSO.parquet
  DIM_UBIGEO.parquet
  FACT_FORMULARIO_SISMEPRE.parquet
  FACT_INGRESO.parquet
  FACT_RENAMU.parquet


In [4]:
dim_tiempo = spark.read.parquet(f"{silver_path}/DIM_TIEMPO.parquet")
print(f"DIM_TIEMPO: {dim_tiempo.count()} filas")
dim_tiempo.select("IdTiempo", "ANIO", "MES").orderBy("IdTiempo").show(10)

DIM_TIEMPO: 67 filas
+--------+----+----+
|IdTiempo|ANIO| MES|
+--------+----+----+
|       0|   0|   0|
|    2001|2001|NULL|
|    2007|2007|NULL|
|    2011|2011|NULL|
|    2012|2012|NULL|
|    2013|2013|NULL|
|    2014|2014|NULL|
|    2015|2015|NULL|
|    2016|2016|NULL|
|    2017|2017|NULL|
+--------+----+----+
only showing top 10 rows


In [5]:
dim_ejecutora = spark.read.parquet(f"{silver_path}/DIM_EJECUTORA.parquet")
print(f"DIM_EJECUTORA: {dim_ejecutora.count()} filas")
duplicados = dim_ejecutora.groupBy("SEC_EJEC").count().filter(F.col("count") > 1)
print(f"Duplicados por SEC_EJEC: {duplicados.count()}")

DIM_EJECUTORA: 1769 filas
Duplicados por SEC_EJEC: 0


In [6]:
fact_ingreso = spark.read.parquet(f"{silver_path}/FACT_INGRESO.parquet")
print(f"FACT_INGRESO: {fact_ingreso.count()} filas")
fact_ingreso.printSchema()

FACT_INGRESO: 2431820 filas
root
 |-- IdTiempo: integer (nullable = true)
 |-- IdNivelGobierno: integer (nullable = true)
 |-- IdSector: integer (nullable = true)
 |-- IdPliego: integer (nullable = true)
 |-- IdEjecutora: integer (nullable = true)
 |-- IdUbigeo: integer (nullable = true)
 |-- IdRubro: integer (nullable = true)
 |-- IdTipoRecurso: integer (nullable = true)
 |-- IdGenerica: integer (nullable = true)
 |-- IdEspecifica: integer (nullable = true)
 |-- MONTO_PIA: long (nullable = true)
 |-- MONTO_PIM: long (nullable = true)
 |-- MONTO_RECAUDADO: decimal(18,2) (nullable = true)



In [7]:
spark.sql(f"""
  SELECT dt.ANIO, SUM(fi.MONTO_RECAUDADO) as TotalRecaudado
  FROM parquet.`{silver_path}/FACT_INGRESO.parquet` fi
  JOIN parquet.`{silver_path}/DIM_TIEMPO.parquet` dt ON fi.IdTiempo = dt.IdTiempo
  GROUP BY dt.ANIO ORDER BY dt.ANIO
""").show()

+----+--------------+
|ANIO|TotalRecaudado|
+----+--------------+
|2021|38905182522.68|
|2022|43029388603.63|
|2023|39890404877.91|
|2024|42006267911.10|
+----+--------------+



In [8]:
# ── Celda 1: Carga de fact_rECUTORA ─────────────────────────────────────────
from pathlib import Path
import pandas as pd

silver_path = Path("data/silver")  # ajusta si tu bootstrap ya la define

fact_r = spark.read.parquet(str(silver_path / "FACT_RENAMU.parquet"))
fact_r.createOrReplaceTempView("fact_renamu")



In [9]:
analisis = spark.sql("""
    SELECT count(1) from fact_renamu
""")

analisis.show(truncate = False)


+--------+
|count(1)|
+--------+
|12770291|
+--------+



In [10]:
dimej = spark.read.parquet(str(silver_path / "DIM_EJECUTORA.parquet"))
dimej.createOrReplaceTempView("dim_ejecutora")
analisis2 = spark.sql("""
    SELECT ejecutora_nombre, count(*) as total
    FROM dim_ejecutora
    GROUP BY ejecutora_nombre
    HAVING count(*) > 1""")
analisis2.show(truncate = False)

+----------------+-----+
|ejecutora_nombre|total|
+----------------+-----+
+----------------+-----+

